# KT-01 - MLP với PyTorch: dự đoán khách hàng rời bỏ (Telco Churn)


## Bước 0. Cấu hình chung

In [3]:
import copy, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for d in ["notebooks", "src", "models", "reports"]:
    (ROOT / d).mkdir(exist_ok=True)

CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
DATA = next(p for p in [ROOT / "data" / CSV, ROOT / CSV, Path("/mnt/user-data/uploads") / CSV] if p.exists())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_all(s=SEED):
    np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

## Bước 2-4. Viết Dataset, MLP, vòng lặp huấn luyện thành module `src/`


In [4]:
(ROOT / "src" / "dataset.py").write_text('''import torch
from torch.utils.data import Dataset

class ChurnDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)   # (N,) -> (N,1) khớp output model
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]
''', encoding="utf-8")

(ROOT / "src" / "model.py").write_text('''import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1),            # logit, không sigmoid
        )
    def forward(self, x):
        return self.net(x)
''', encoding="utf-8")

(ROOT / "src" / "train.py").write_text('''import copy
import numpy as np, torch
from sklearn.metrics import average_precision_score

def predict(model, loader, device, eval_mode=True):
    # eval_mode=False chỉ dùng cho thí nghiệm "quên model.eval()"
    model.train(not eval_mode)
    ps, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            ps.append(torch.sigmoid(model(xb.to(device))).cpu())   # sigmoid ở bước predict
            ys.append(yb)
    return torch.cat(ps).numpy().ravel(), torch.cat(ys).numpy().ravel()

def fit(model, train_loader, val_loader, optimizer, criterion, device,
        epochs=100, patience=10, zero_grad=True):
    hist = {"train_loss": [], "val_loss": [], "val_prauc": []}
    best, best_state, wait = -1.0, None, 0
    for ep in range(epochs):
        model.train()                                    # bật Dropout / BatchNorm train
        tl, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            if zero_grad:
                optimizer.zero_grad()                    # nếu thiếu: gradient cộng dồn qua các batch
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            tl += loss.item() * len(xb); n += len(xb)

        model.eval()                                     # tắt Dropout, BatchNorm dùng thống kê chạy
        vl, vn, ps, ys = 0.0, 0, [], []
        with torch.no_grad():                            # không tính gradient khi đánh giá
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                vl += criterion(out, yb).item() * len(xb); vn += len(xb)
                ps.append(torch.sigmoid(out).cpu()); ys.append(yb.cpu())
        p = np.nan_to_num(torch.cat(ps).numpy().ravel(), nan=0.5)    # tránh lỗi khi loss nổ thành NaN
        pr = average_precision_score(torch.cat(ys).numpy().ravel(), p)
        hist["train_loss"].append(tl / n); hist["val_loss"].append(vl / vn); hist["val_prauc"].append(pr)

        if pr > best:                                    # early stopping thủ công theo PR-AUC val
            best, wait, best_state = pr, 0, copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)                # trả về trọng số tốt nhất (checkpoint)
    return hist
''', encoding="utf-8")

sys.path.insert(0, str(ROOT / "src"))
from dataset import ChurnDataset
from model import MLP
from train import fit, predict

## Bước 1. Tiền xử lý bằng sklearn Pipeline


In [5]:
df = pd.read_csv(DATA)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")   # 11 dòng khoảng trắng -> NaN
y = (df.pop("Churn") == "Yes").astype(int).values
df = df.drop(columns="customerID")

num_cols = df.select_dtypes("number").columns.tolist()
cat_cols = df.select_dtypes("object").columns.tolist()
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

Xtr_raw, Xtmp, ytr, ytmp = train_test_split(df, y, test_size=0.3, stratify=y, random_state=SEED)
Xva_raw, Xte_raw, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.5, stratify=ytmp, random_state=SEED)
Xtr = prep.fit_transform(Xtr_raw).astype(np.float32)
Xva = prep.transform(Xva_raw).astype(np.float32)
Xte = prep.transform(Xte_raw).astype(np.float32)
n_feat = Xtr.shape[1]
print(Xtr.shape, Xva.shape, Xte.shape, "ty le churn train:", ytr.mean().round(3))

(4930, 45) (1056, 45) (1057, 45) ty le churn train: 0.265


## Bước 2-3. DataLoader (in thử 1 batch) và đếm số tham số

In [6]:
def loader(X, y, shuffle=False, bs=128):
    return DataLoader(ChurnDataset(X, y), batch_size=bs, shuffle=shuffle, drop_last=shuffle)  # drop_last tránh batch 1 mẫu làm BatchNorm lỗi

tr_ld, va_ld, te_ld = loader(Xtr, ytr, True), loader(Xva, yva), loader(Xte, yte)
xb, yb = next(iter(tr_ld))
print("batch:", xb.shape, yb.shape)

n_params = sum(p.numel() for p in MLP(n_feat).parameters())
print("so tham so:", n_params)

batch: torch.Size([128, 45]) torch.Size([128, 1])
so tham so: 14593


## Bước 4 + 9. Huấn luyện với 3 optimizer (SGD / Adam / AdamW)


In [7]:
spw = float((ytr == 0).sum() / (ytr == 1).sum())          # trọng số lớp dương, xử lý lệch nhãn
pw = torch.tensor([spw])

OPT = {
    "SGD":   lambda p: torch.optim.SGD(p, lr=1e-2, momentum=0.9),
    "Adam":  lambda p: torch.optim.Adam(p, lr=1e-3),
    "AdamW": lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=1e-4),
}

def run(name, dev=device, **kw):
    seed_all()
    m = MLP(n_feat).to(dev)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw.to(dev))
    t = time.time()
    h = fit(m, tr_ld, va_ld, OPT[name](m.parameters()), crit, dev, **kw)
    return m, h, time.time() - t

results = {}
for name in OPT:
    m, h, t = run(name)
    p, tgt = predict(m, te_ld, device)
    results[name] = dict(model=m, hist=h, time=t, epochs=len(h["val_prauc"]),
                         val=max(h["val_prauc"]), test=average_precision_score(tgt, p))
    print(name, {k: round(v, 4) for k, v in results[name].items() if k in ("val", "test", "time")}, "epochs:", results[name]["epochs"])

best_name = max(results, key=lambda k: results[k]["val"])  # chọn theo val, không chọn theo test
best = results[best_name]
print("optimizer tot nhat (val):", best_name)

SGD {'time': 17.0389, 'val': 0.6313, 'test': 0.654} epochs: 20
Adam {'time': 19.0078, 'val': 0.6286, 'test': 0.6508} epochs: 18
AdamW {'time': 19.1617, 'val': 0.6288, 'test': 0.6433} epochs: 20
optimizer tot nhat (val): SGD


## Bước 7 + 9. Learning curve và biểu đồ so sánh optimizer

In [8]:
h = best["hist"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h["train_loss"], label="train"); ax[0].plot(h["val_loss"], label="val")
ax[0].set(title=f"Loss theo epoch ({best_name})", xlabel="epoch", ylabel="BCEWithLogits"); ax[0].legend()
ax[1].plot(h["val_prauc"], color="tab:green")
ax[1].set(title="PR-AUC val theo epoch", xlabel="epoch", ylabel="PR-AUC")
plt.tight_layout(); plt.savefig(ROOT / "reports" / "learning_curve.png", dpi=130); plt.close()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k, v in results.items():
    ax[0].plot(v["hist"]["val_prauc"], label=k)
ax[0].set(title="PR-AUC val theo epoch", xlabel="epoch", ylabel="PR-AUC"); ax[0].legend()
ax[1].bar(list(results), [v["test"] for v in results.values()], color="tab:blue")
ax[1].set(title="PR-AUC test", ylim=(min(v["test"] for v in results.values()) - 0.05, None))
plt.tight_layout(); plt.savefig(ROOT / "reports" / "optimizer_comparison.png", dpi=130); plt.close()

## Bước 8 + 11. Lưu checkpoint tốt nhất bằng `state_dict`, nạp lại và kiểm tra cùng kết quả

In [9]:
torch.save(best["model"].state_dict(), ROOT / "models" / "best.pt")

m2 = MLP(n_feat).to(device)
m2.load_state_dict(torch.load(ROOT / "models" / "best.pt", map_location=device))
m2.eval()
p1, _ = predict(best["model"], te_ld, device)
p2, _ = predict(m2, te_ld, device)
reload_ok = bool(np.allclose(p1, p2, atol=1e-6))
print("nap lai cho cung ket qua:", reload_ok)

nap lai cho cung ket qua: True


## Bước 5. Thí nghiệm sai 1: bỏ `optimizer.zero_grad()`


In [10]:
EXP_EP = 30
_, h_ok, _ = run("AdamW", epochs=EXP_EP, patience=EXP_EP)
_, h_bad, _ = run("AdamW", epochs=EXP_EP, patience=EXP_EP, zero_grad=False)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for h_, lb in [(h_ok, "co zero_grad"), (h_bad, "bo zero_grad")]:
    ax[0].plot(h_["train_loss"], label=lb); ax[1].plot(h_["val_prauc"], label=lb)
ax[0].set(title="Train loss", xlabel="epoch"); ax[1].set(title="PR-AUC val", xlabel="epoch"); ax[0].legend()
plt.tight_layout(); plt.savefig(ROOT / "reports" / "exp_zero_grad.png", dpi=130); plt.close()

ex5 = dict(loss_ok=h_ok["train_loss"][-1], loss_bad=h_bad["train_loss"][-1],
           pr_ok=max(h_ok["val_prauc"]), pr_bad=max(h_bad["val_prauc"]))
print(ex5)

{'loss_ok': 0.6525296019880396, 'loss_bad': 0.746688585532339, 'pr_ok': 0.6287861052331439, 'pr_bad': 0.6291371187619683}


## Bước 6. Thí nghiệm sai 2: quên `model.eval()` khi đánh giá


In [11]:
shuf_ld = DataLoader(ChurnDataset(Xva, yva), batch_size=256, shuffle=True)

def repeat_eval(eval_mode, n=20):
    out = []
    for _ in range(n):
        m = copy.deepcopy(best["model"])               # deepcopy vì chế độ train làm đổi thống kê BatchNorm
        p, t = predict(m, shuf_ld, device, eval_mode=eval_mode)
        out.append(average_precision_score(t, p))
    return np.array(out)

s_eval, s_train = repeat_eval(True), repeat_eval(False)
ex6 = dict(eval_mean=s_eval.mean(), eval_std=s_eval.std(), train_mean=s_train.mean(),
           train_std=s_train.std(), train_range=s_train.max() - s_train.min())
print(ex6)

{'eval_mean': np.float64(0.6313121284785432), 'eval_std': np.float64(1.1102230246251565e-16), 'train_mean': np.float64(0.6131083657804566), 'train_std': np.float64(0.008509052462469897), 'train_range': np.float64(0.03595141157990178)}


## Bước 10. So sánh với XGBoost trên cùng dữ liệu đã tiền xử lý

In [12]:
t0 = time.time()
xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8,
                    colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="aucpr", random_state=SEED)
xgb.fit(Xtr, ytr)
t_xgb = time.time() - t0

loc_mlp = sum(len((ROOT / "src" / f).read_text(encoding="utf-8").splitlines()) for f in ["dataset.py", "model.py", "train.py"])
cmp = pd.DataFrame([
    dict(model=f"MLP PyTorch ({best_name})", val_prauc=best["val"], test_prauc=best["test"],
         train_time_s=best["time"], code_lines=loc_mlp),
    dict(model="XGBoost", val_prauc=average_precision_score(yva, xgb.predict_proba(Xva)[:, 1]),
         test_prauc=average_precision_score(yte, xgb.predict_proba(Xte)[:, 1]),
         train_time_s=t_xgb, code_lines=2),            # code_lines: khai báo + fit
]).round(4)
cmp.to_csv(ROOT / "reports" / "vs_xgboost.csv", index=False)
gap = float(cmp.test_prauc[1] - cmp.test_prauc[0])     # >0: XGBoost cao hơn
print(cmp, "\nchenh lech (XGB - MLP):", round(gap, 4))

               model  val_prauc  test_prauc  train_time_s  code_lines
0  MLP PyTorch (SGD)     0.6313      0.6540       17.0389          74
1            XGBoost     0.6405      0.6592        2.1668           2 
chenh lech (XGB - MLP): 0.0052


## Bước 12. Thời gian train CPU vs GPU (10 epoch, không early stopping)

In [13]:
timing = {}
for dev in ["cpu"] + (["cuda"] if torch.cuda.is_available() else []):
    timing[dev] = run("AdamW", dev=torch.device(dev), epochs=10, patience=10)[2]
print(timing)

{'cpu': 2.3961658477783203}


## Sinh `requirements.txt` và `README.md` (các bước làm + nhận xét theo số liệu thật)

In [14]:
(ROOT / "requirements.txt").write_text("torch\nnumpy\npandas\nscikit-learn\nxgboost\nmatplotlib\n", encoding="utf-8")

opt_rows = "\n".join(f"| {k} | {v['epochs']} | {v['val']:.4f} | {v['test']:.4f} | {v['time']:.1f} |" for k, v in results.items())
gpu = f"CPU {timing['cpu']:.1f}s, GPU {timing['cuda']:.1f}s" if "cuda" in timing else f"chỉ có CPU: {timing['cpu']:.1f}s (không có GPU để so sánh)"
winner = "XGBoost cao hơn MLP" if gap > 0 else "MLP cao hơn hoặc bằng XGBoost"

readme = f"""# KT-01 - MLP với PyTorch: dự đoán khách hàng rời bỏ

Dữ liệu: Telco Customer Churn (7.043 dòng, nhãn lệch). Chia 70/15/15 có stratify, số đặc trưng sau one-hot: {n_feat}, số tham số MLP: {n_params:,}.

## Các bước thực hiện và nhận xét

**1. Tiền xử lý.** Ép kiểu `TotalCharges` (11 dòng trống thành NaN, điền median), bỏ `customerID`, chuẩn hóa biến số, one-hot biến phân loại, tất cả trong sklearn Pipeline chỉ fit trên tập train.

**2. Dataset và DataLoader.** `ChurnDataset` trả về `(x, y)` với `y` đã `unsqueeze(1)`. Batch mẫu có shape x = {tuple(xb.shape)}, y = {tuple(yb.shape)}.

**3. Mô hình.** MLP 128-64-1 có BatchNorm và Dropout; lớp cuối là logit, không có sigmoid. Dùng `BCEWithLogitsLoss(pos_weight={spw:.2f})` để xử lý lệch lớp.

**4. Vòng lặp huấn luyện.** `src/train.py` có đủ `zero_grad`, `train()`, `eval()`, `no_grad`. Chỉ số theo dõi là PR-AUC vì nhãn lệch.

**5. Thí nghiệm sai 1: bỏ `zero_grad()`.** Sau {EXP_EP} epoch, train loss là {ex5['loss_ok']:.4f} khi đúng và {ex5['loss_bad']:.4f} khi bỏ; PR-AUC val tốt nhất lần lượt {ex5['pr_ok']:.4f} và {ex5['pr_bad']:.4f}. Lý do: gradient của các batch cộng dồn, hướng cập nhật bị nhiễm gradient cũ nên đường loss kém ổn định và model khó hội tụ. Xem `reports/exp_zero_grad.png`.

**6. Thí nghiệm sai 2: bỏ `model.eval()`.** Đánh giá 20 lần cùng một model: chế độ eval cho PR-AUC {ex6['eval_mean']:.4f} (độ lệch chuẩn {ex6['eval_std']:.5f}); chế độ train cho {ex6['train_mean']:.4f} (độ lệch chuẩn {ex6['train_std']:.5f}, biên độ {ex6['train_range']:.4f}). Dropout còn bật và BatchNorm dùng thống kê của từng batch nên điểm dao động ngẫu nhiên, đồng thời thường lệch thấp hơn.

**7. Learning curve.** `reports/learning_curve.png` (optimizer {best_name}). Train loss và val loss cho thấy mức độ overfit; PR-AUC val dùng để chọn epoch tốt nhất.

**8. Early stopping và checkpoint.** Dừng khi PR-AUC val không cải thiện sau 10 epoch, trả về trọng số tốt nhất và lưu `models/best.pt` bằng `state_dict`.

**9. So sánh 3 optimizer.** Chọn theo PR-AUC val, không chọn theo test. Xem `reports/optimizer_comparison.png`.

| Optimizer | Số epoch | PR-AUC val | PR-AUC test | Thời gian (s) |
|---|---|---|---|---|
{opt_rows}

Optimizer được chọn: **{best_name}**.

**10. So sánh với XGBoost** (`reports/vs_xgboost.csv`).

| Mô hình | PR-AUC val | PR-AUC test | Thời gian train (s) | Số dòng code |
|---|---|---|---|---|
""" + "\n".join(f"| {r.model} | {r.val_prauc} | {r.test_prauc} | {r.train_time_s} | {r.code_lines} |" for r in cmp.itertuples()) + f"""

Kết luận: trên PR-AUC test, {winner} (chênh lệch XGBoost trừ MLP = {gap:+.4f}). Tiêu chí "không thua XGBoost quá 0,03": {'đạt' if gap <= 0.03 else 'không đạt'}. Với dữ liệu bảng nhỏ, MLP cần nhiều code và nhiều bước tinh chỉnh hơn nhưng không mang lại lợi thế rõ so với mô hình cây; XGBoost là lựa chọn thực dụng hơn cho bài toán này. Giá trị của bài là hiểu vòng lặp huấn luyện để dùng cho các kiến trúc sau.

**11. Lưu và nạp model.** Lưu `state_dict`, nạp vào MLP mới, gọi `eval()`. Xác suất dự đoán trùng khớp với model gốc: {reload_ok}.

**12. CPU và GPU.** 10 epoch: {gpu}. Với dữ liệu nhỏ như thế này, GPU thường không nhanh hơn đáng kể vì chi phí chuyển dữ liệu và khởi chạy chiếm phần lớn.

## Cấu trúc thư mục
`notebooks/mlp_pytorch_churn.ipynb`, `src/{{dataset,model,train}}.py`, `models/best.pt`, `reports/{{learning_curve.png, optimizer_comparison.png, exp_zero_grad.png, vs_xgboost.csv}}`, `requirements.txt`.
"""
(ROOT / "README.md").write_text(readme, encoding="utf-8")
print("Da sinh README.md, requirements.txt, models/best.pt, reports/*")

Da sinh README.md, requirements.txt, models/best.pt, reports/*
